# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s. We'll enumerate the record set `@id`s and show the field `@id`s they contain.

In [ ]:
# List all record sets by their '@id'
record_sets = []
for rs in metadata.record_sets:
    print(f"RecordSet name: {rs.name}, @id: {rs.id}")
    record_sets.append(rs.id)
    field_ids = [f.id for f in rs.fields]
    print(f"  Fields: {field_ids}")
if not record_sets:
    print("No record sets were found in the Croissant schema. Exiting.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# If no record set, stop here
if not record_sets:
    raise ValueError("No record sets defined in the Croissant schema.")

# Extract data from each record set
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set: {record_set_id}, with {len(df)} records and columns: {df.columns.tolist()}")

# We'll select the first available record set for demonstration
selected_record_set_id = record_sets[0]
print(f"\nPreview of data from record set {selected_record_set_id}:")
display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Below are example EDA operations:
- Filtering records based on a numeric field
- Normalizing the numeric field
- Optionally grouping by a categorical field

All entities are referenced by their `@id`.

In [ ]:
df = dataframes[selected_record_set_id]

# Try to auto-detect numeric fields from the columns
numeric_field_candidates = []
for col in df.columns:
    # Try convert to numeric
    try:
        vals = pd.to_numeric(df[col], errors='coerce')
        # Heuristic: if at least 80% are number
        if (vals.notna().mean() > 0.8):
            numeric_field_candidates.append(col)
    except Exception:
        continue

if not numeric_field_candidates:
    print("No numeric fields detected in the selected record set. Please check the schema/fields manually.")
else:
    numeric_field_id = numeric_field_candidates[0]
    print(f"Using numeric field for analysis: {numeric_field_id}")
    # Try to filter
    vals = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = vals.mean() if vals.notna().sum() else 10
    filtered_df = df[vals > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_vals = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_vals - filtered_vals.mean()) / filtered_vals.std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try to group by a categorical field (heuristically pick first 'str' column not equal to id or numeric)
    group_field = None
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == object:
            group_field = col
            break
    # Only include grouping if a field exists
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[[numeric_field_id]].mean()
        print(f"Grouped data by {group_field}, mean of {numeric_field_id}:")
        display(grouped_df.head())

## 5. Visualization

Visualize data distributions and relationships between selected fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_candidates:
    plt.figure(figsize=(8, 5))
    plt.title(f"Distribution of {numeric_field_id}")
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), kde=True)
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group_field chosen, boxplot per category
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, we:
- Loaded the FAIR^2 dataset schema using `mlcroissant`
- Inspected available record sets and their fields by `@id`
- Extracted tabular data for exploration and filtering
- Demonstrated numeric field normalization and categorical grouping
- Visualized field distributions where possible

For further processing, consult the Croissant schema and explore additional fields and relationships using their `@id`s.